# Minimal Ω Structural Concentration Test

Upload any CSV and test whether an independently defined binary event concentrates in high-Ω states.

This is a structural concentration test, not prediction. The event must be independent from Ω. Thresholds are fixed ex-ante. Null results are valid.

In [ ]:
import numpy as np
import pandas as pd

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WINDOW = 20
HIGH_OMEGA_TOP_FRACTION = 0.01

print("Parameters fixed ex-ante:")
print(f"  rolling std window = {WINDOW}")
print(f"  high Ω = top {HIGH_OMEGA_TOP_FRACTION:.0%}")

## Upload CSV

In [ ]:
if IN_COLAB:
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
else:
    csv_path = input("CSV path: ").strip()

df = pd.read_csv(csv_path)
print(f"Loaded {csv_path}")
print(f"Rows: {len(df)}")
print("Columns:")
for i, col in enumerate(df.columns):
    print(f"  [{i}] {col}")

df.head()

## Select Columns

The event column must be binary and defined independently from Ω.

In [ ]:
def choose_column(prompt, required=True):
    raw = input(prompt).strip()
    if not raw and not required:
        return None
    if raw.isdigit():
        return df.columns[int(raw)]
    return raw

value_col = choose_column("Value column name or index: ")
time_col = choose_column("Optional time/order column name or index; press Enter to skip: ", required=False)
event_col = choose_column("Binary event column name or index: ")

print("Selected:")
print(f"  value column: {value_col}")
print(f"  time/order column: {time_col}")
print(f"  event column: {event_col}")

## Compute Ω And Concentration

In [ ]:
work = df.copy()

if time_col is not None:
    work = work.sort_values(time_col).reset_index(drop=True)

work["_value"] = pd.to_numeric(work[value_col], errors="coerce")
work["_event"] = pd.to_numeric(work[event_col], errors="coerce")

bad_events = sorted(set(work["_event"].dropna().unique()) - {0, 1})
if bad_events:
    raise ValueError(f"Event column must be binary 0/1. Found other values: {bad_events}")

work["I"] = work["_value"].rolling(WINDOW).std()
work["G"] = work["_value"].diff().abs()
work["Omega"] = work["I"] * work["G"]

valid = work.dropna(subset=["Omega", "_event"]).copy()
if valid.empty:
    raise ValueError("No valid rows after computing Ω. Try a CSV with at least 20 numeric value rows.")

threshold = valid["Omega"].quantile(1 - HIGH_OMEGA_TOP_FRACTION)
valid["high_Omega"] = valid["Omega"] >= threshold

n_rows = int(len(valid))
n_high = int(valid["high_Omega"].sum())
n_event_high = int(valid.loc[valid["high_Omega"], "_event"].sum())

p_event_high = valid.loc[valid["high_Omega"], "_event"].mean() if n_high else np.nan
baseline_p_event = valid["_event"].mean()
ratio = p_event_high / baseline_p_event if baseline_p_event > 0 else np.nan

print("Preview of computed columns:")
display_cols = [c for c in [time_col, value_col, event_col] if c is not None]
display(valid[display_cols + ["I", "G", "Omega", "high_Omega"]].tail(10))

## Final Copy & Paste Block

In [ ]:
domain = input("Domain label for this test: ").strip() or csv_path

print("COPY & PASTE")
print("-----------")
print(f"Domain: {domain}")
print(f"P(event | high Ω): {p_event_high:.6g}")
print(f"Baseline P(event): {baseline_p_event:.6g}")
print(f"Ratio: {ratio:.6g}")
print(f"n_rows: {n_rows}")
print(f"n_high: {n_high}")
print(f"n_event_high: {n_event_high}")